In [ ]:
import networkx as nx
import polars as pl
import matplotlib.pyplot as plt

In [15]:
# 1) Load the dataset from the specific year in Parquet format: 
path = "../data/processed/trade_2024.parquet"
df = pl.read_parquet(path)
df

Year,Exporter,Importer,Product_Code,Value_Thousands_USD
u16,cat,cat,u32,f64
2024,"""AFG""","""AGO""",80810,0.176
2024,"""AFG""","""AGO""",330499,2.295
2024,"""AFG""","""AGO""",732510,5.617
2024,"""AFG""","""AGO""",848330,2.42
2024,"""AFG""","""AGO""",853610,0.605
…,…,…,…,…
2024,"""ZMB""","""URY""",610429,0.585
2024,"""ZMB""","""URY""",848490,0.048
2024,"""ZMB""","""URY""",870810,0.02


In [16]:
# 2) Select a specific product (for example, petrol with code 2709): 
df = df.filter(pl.col("Product_Code") == 270900)
df

Year,Exporter,Importer,Product_Code,Value_Thousands_USD
u16,cat,cat,u32,f64
2024,"""ALB""","""CHN""",270900,1.582
2024,"""ALB""","""GRC""",270900,30.745
2024,"""ALB""","""ITA""",270900,10112.894
2024,"""ALB""","""ESP""",270900,98322.135
2024,"""DZA""","""AUS""",270900,96921.278
…,…,…,…,…
2024,"""VEN""","""ESP""",270900,1.2732e6
2024,"""VEN""","""USA""",270900,5.8820e6
2024,"""WSM""","""ASM""",270900,0.182


In [17]:
# 3) Group by country pairs to add up the total value of trade and prevent Networkx from overlapping each edge: 
df = df.group_by(["Exporter", "Importer"]).agg(
    pl.col("Value_Thousands_USD").sum().alias("weight")
)
df

Exporter,Importer,weight
cat,cat,f64
"""COG""","""SGP""",221782.761
"""ECU""","""PAN""",5.3062e6
"""ARE""","""DEU""",600854.139
"""OMN""","""NLD""",0.236
"""QAT""","""NLD""",26.611
…,…,…
"""USA""","""BHS""",11.304
"""SGP""","""THA""",2310.928
"""ARG""","""AUS""",47551.452


In [18]:
# 4) Create the graph as a directed graph (Exporter -> Importer) and the edges weighted by its trade value:
graph = nx.from_pandas_edgelist(
    df.to_pandas(),
    source="Exporter",
    target="Importer",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

In [19]:
# 5) Use the Gephi software to visualize in local the graph: 
nx.write_gexf(graph, "world_trade_network_petrol.gexf")